# Recomender

In [1]:
"""
Strengths-based occupational recommender: proof of concept.

Runs cosine similarity between a candidate strengths vector and a set of
occupation skill vectors, and prints ranked recommendations with
per-dimension explanations. Reports demand as an independent signal
alongside similarity, not combined into a single score.
"""

import math



DIMENSIONS = [
    "Analytical thinking",
    "Creative thinking",
    "Systems thinking",
    "Resilience / flexibility",
    "Empathy & active listening",
    "Communication (oral)",
    "Communication (written)",
    "Leadership & social influence",
    "Service orientation",
    "Technological literacy",
]

# Occupation vectors anchored in O*NET Skills Importance ratings (rescaled
# 0-100 -> 0-1). Each dimension is the max of its mapped O*NET skills.
# Mapping documented in onet_mapping.md; computation in compute_vectors.py.
# Resilience assigned from BLS narrative (O*NET has no direct Skills entry).
# One documented adjustment: Content Creator Creative raised from 0.19 to 0.75
# to reflect Indonesian entry-level content creator context.

OCCUPATIONS = {
    "Customer Service Rep":        [0.56, 0.22, 0.31, 0.70, 0.75, 0.72, 0.60, 0.50, 0.75, 0.22],
    "Retail Sales Assistant":      [0.53, 0.22, 0.25, 0.65, 0.69, 0.69, 0.50, 0.72, 0.69, 0.25],
    "Administrative Assistant":    [0.50, 0.22, 0.25, 0.55, 0.75, 0.75, 0.72, 0.41, 0.63, 0.22],
    "Content Creator":             [0.66, 0.75, 0.35, 0.60, 0.72, 0.72, 0.94, 0.60, 0.53, 0.19],
    "IT Support Specialist":       [0.69, 0.35, 0.50, 0.65, 0.75, 0.75, 0.75, 0.50, 0.53, 0.50],
    "Food Service Worker":         [0.47, 0.00, 0.25, 0.75, 0.63, 0.63, 0.44, 0.44, 0.72, 0.25],
    "Software Developer":          [0.72, 0.60, 0.63, 0.55, 0.60, 0.53, 0.63, 0.47, 0.47, 0.75],
}

# BLS 2024-2034 Employment Projections. Growth is percent change in
# employment 2024-2034; openings is annual average occupational openings
# 2024-2034. All figures are for the specific SOC code identified for each
# occupation. Source: BLS Table 1.2 Occupational projections, 2024-2034
# (https://www.bls.gov/emp/tables/occupational-projections-and-characteristics.htm),
# last updated 28 August 2025. Growth percentages are rounded to the
# nearest whole percent for reporting.
DEMAND = {
    # 43-4051 Customer Service Representatives: -5.5% -> -5%, 341,700
    "Customer Service Rep":      {"growth_pct": -5.0,  "annual_openings": 341700},
    # 41-2031 Retail Salespersons: -0.5% -> -1%, 555,800
    "Retail Sales Assistant":    {"growth_pct": -1.0,  "annual_openings": 555800},
    # 43-6014 Secretaries and Administrative Assistants, Except Legal,
    # Medical, and Executive: -1.6% -> -2%, 202,800
    "Administrative Assistant":  {"growth_pct": -2.0,  "annual_openings": 202800},
    # 27-3043 Writers and Authors: 3.6% -> +4%, 13,400
    "Content Creator":           {"growth_pct":  4.0,  "annual_openings": 13400},
    # 15-1232 Computer User Support Specialists: -3.7% -> -4%, 40,800
    "IT Support Specialist":     {"growth_pct": -4.0,  "annual_openings": 40800},
    # 35-3031 Waiters and Waitresses: -0.7% -> -1%, 456,700
    "Food Service Worker":       {"growth_pct": -1.0,  "annual_openings": 456700},
    # 15-1252 Software Developers: 15.8% -> +16%, 115,200
    "Software Developer":        {"growth_pct": 16.0,  "annual_openings": 115200},
}

# Candidate A (synthetic, illustrative) - analytical/technical strengths profile
CANDIDATE_A = [0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.80, 0.35, 0.45, 0.80]

# Candidate B (synthetic, illustrative) - legacy person-oriented strengths
# profile. Retained for backward compatibility with earlier iterations of the
# recommender. THE PAPER'S ILLUSTRATIVE EXAMPLE USES THE DEAF CANDIDATE B
# DEFINED IN candidate_b_deaf.py (CANDIDATE_B_DEAF and CANDIDATE_B_SUNYI),
# not this vector.
CANDIDATE_B_LEGACY = [0.40, 0.50, 0.40, 0.70, 0.90, 0.80, 0.55, 0.50, 0.85, 0.45]


def l2_normalize(v):
    """Rescale a vector so it has length 1. This lets cosine similarity focus
    on the shape of the strengths profile rather than its overall magnitude."""
    magnitude = math.sqrt(sum(x * x for x in v))
    if magnitude == 0:
        return v
    return [x / magnitude for x in v]


def cosine_similarity(a, b):
    """Cosine of the angle between two vectors. Returns a value in [0, 1] for
    non-negative vectors. Higher means the strengths profiles have their high
    values in similar places."""
    a_norm = l2_normalize(a)
    b_norm = l2_normalize(b)
    return sum(x * y for x, y in zip(a_norm, b_norm))


def per_dimension_contributions(candidate, occupation):
    """Break the cosine similarity down by dimension so we can explain which
    dimensions drove the match."""
    c_norm = l2_normalize(candidate)
    o_norm = l2_normalize(occupation)
    contributions = []
    for i, dim in enumerate(DIMENSIONS):
        contributions.append((dim, c_norm[i] * o_norm[i]))
    return sorted(contributions, key=lambda x: -x[1])


def min_max_scale(values):
    """Rescale a list of numbers to sit between 0 and 1 across the sample."""
    lo, hi = min(values), max(values)
    if hi == lo:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


def demand_score(demand_data):
    """Combine growth and openings into a single demand score with equal
    weighting. Used as a secondary contextual signal, not for ranking."""
    occ_names = list(demand_data.keys())
    growths = [demand_data[o]["growth_pct"] for o in occ_names]
    openings = [demand_data[o]["annual_openings"] for o in occ_names]
    g_scaled = min_max_scale(growths)
    a_scaled = min_max_scale(openings)
    return {occ_names[i]: 0.5 * g_scaled[i] + 0.5 * a_scaled[i] for i in range(len(occ_names))}


def recommend(candidate, occupations, demand_data, top_k=None, show_contribs=3):
    """Produce ranked recommendations for a candidate."""
    scores = []
    for name, vec in occupations.items():
        score = cosine_similarity(candidate, vec)
        scores.append((name, score, vec))
    scores.sort(key=lambda x: -x[1])

    demand = demand_score(demand_data)

    print("=" * 78)
    print(f"{'Rank':<6}{'Occupation':<28}{'Skill match':<14}{'Demand':<10}")
    print("-" * 78)
    for rank, (name, score, vec) in enumerate(scores[:top_k] if top_k else scores, 1):
        d = demand[name]
        print(f"{rank:<6}{name:<28}{score:<14.3f}{d:<10.3f}")
    print("=" * 78)
    print()

    # Explanations for the top 3
    print("Why each of the top three matches:")
    print("-" * 78)
    for rank, (name, score, vec) in enumerate(scores[:3], 1):
        contribs = per_dimension_contributions(candidate, vec)
        top_dims = contribs[:show_contribs]
        print(f"\n{rank}. {name} (skill match = {score:.3f})")
        print(f"   Top contributing dimensions:")
        for dim, contrib in top_dims:
            print(f"     - {dim}: {contrib:.3f}")

    print()
    print("Demand context (BLS 2024-2034 projections):")
    print("-" * 78)
    for name, _, _ in scores:
        d = demand_data[name]
        print(f"  {name}: {d['growth_pct']:+.0f}% growth, ~{d['annual_openings']:,} annual openings")


def show_candidate(name, description, vector):
    print(f"\n{name}: {description}")
    print("Strengths profile:")
    for dim, val in zip(DIMENSIONS, vector):
        bar = "#" * int(val * 20)
        print(f"  {dim:<32} {val:.2f}  {bar}")
    print()


if __name__ == "__main__":
    print("\nStrengths-based occupational recommender: proof of concept")
    print("=" * 78)

    show_candidate(
        "Candidate A",
        "final-year undergraduate, analytical/technical strengths",
        CANDIDATE_A,
    )
    recommend(CANDIDATE_A, OCCUPATIONS, DEMAND)

    print("\n\n")

    # The paper's Candidate B (Deaf SMK graduate, hospitality track) with both
    # scorings is defined and run in candidate_b_deaf.py. Run that script to
    # reproduce the Candidate B rankings reported in Section IV.
    #
    # The legacy person-oriented profile below is retained only for backward
    # compatibility with earlier iterations. It is NOT the paper's Candidate B
    # and its scores do not match the ones reported in Section IV.
    print("\nFor the paper's Candidate B (Deaf SMK graduate, hearing-default")
    print("and Sunyi-inclusive scorings), run: python candidate_b_deaf.py\n")

    show_candidate(
        "Legacy person-oriented profile (NOT the paper's Candidate B)",
        "retained for backward compatibility; see candidate_b_deaf.py for the paper's B",
        CANDIDATE_B_LEGACY,
    )
    recommend(CANDIDATE_B_LEGACY, OCCUPATIONS, DEMAND)



Strengths-based occupational recommender: proof of concept

Candidate A: final-year undergraduate, analytical/technical strengths
Strengths profile:
  Analytical thinking              0.85  #################
  Creative thinking                0.55  ###########
  Systems thinking                 0.75  ###############
  Resilience / flexibility         0.50  ##########
  Empathy & active listening       0.55  ###########
  Communication (oral)             0.45  #########
  Communication (written)          0.80  ################
  Leadership & social influence    0.35  #######
  Service orientation              0.45  #########
  Technological literacy           0.80  ################

Rank  Occupation                  Skill match   Demand    
------------------------------------------------------------------------------
1     Software Developer          0.989         0.594     
2     IT Support Specialist       0.948         0.049     
3     Content Creator             0.899         0.21

# Candidate B Deaf Adjustment Recomender

In [2]:
"""
Run recommender on Candidate B redefined as an explicit Deaf SMK student
in a hospitality track, trained in a Sunyi Coffee-style inclusive workplace model.

Strengths scored honestly on the ten dimensions as designed:
- Oral communication is spoken-language framed by O*NET's Speaking skill.
  For a Deaf candidate whose primary language is BISINDO, this dimension
  is scored low (0.30) - not because they lack communication ability,
  but because the DIMENSION as constructed measures spoken language.
- Empathy and active listening scored high (0.85) - visual attention,
  community-oriented listening, facial-cue reading are real strengths.
- The gap between "cannot speak Indonesian fluently" and "has strong
  interpersonal communication capacity" is exactly what the recommender's
  current dimensions do not capture. That gap is the point.
"""

# Candidate B redefined - Deaf SMK graduate, hospitality/service track,
# trained in Sunyi Coffee-style inclusive workplace
CANDIDATE_B_DEAF = [
    0.45,  # Analytical thinking
    0.55,  # Creative thinking
    0.40,  # Systems thinking
    0.85,  # Resilience/flexibility - high, real
    0.85,  # Empathy & active listening - high (visual, community-oriented)
    0.30,  # Communication (oral) - low: spoken Indonesian limited; sign primary
    0.55,  # Communication (written) - moderate
    0.45,  # Leadership & social influence
    0.85,  # Service orientation - high
    0.45,  # Technological literacy
]


def l2(v):
    m = math.sqrt(sum(x * x for x in v))
    return v if m == 0 else [x / m for x in v]


def cosine(a, b):
    an, bn = l2(a), l2(b)
    return sum(x * y for x, y in zip(an, bn))


def per_dim_contributions(cand, occ_vec):
    """Cosine similarity broken down per dimension."""
    cn = l2(cand)
    on = l2(occ_vec)
    return [c * o for c, o in zip(cn, on)]


# Run the ranking
print("=" * 78)
print("CANDIDATE B (Deaf) - strengths on the ten dimensions")
print("=" * 78)
for dim, val in zip(DIMENSIONS, CANDIDATE_B_DEAF):
    print(f"  {dim:<40} {val:.2f}")

print()
print("=" * 78)
print("RANKING")
print("=" * 78)
scored = [(occ, cosine(CANDIDATE_B_DEAF, v)) for occ, v in OCCUPATIONS.items()]
scored.sort(key=lambda x: -x[1])

for i, (occ, score) in enumerate(scored, 1):
    print(f"  {i}. {occ:<28} {score:.3f}")

print()
print("=" * 78)
print("PER-DIMENSION CONTRIBUTIONS FOR TOP 3")
print("=" * 78)
for occ, score in scored[:3]:
    contribs = per_dim_contributions(CANDIDATE_B_DEAF, OCCUPATIONS[occ])
    print(f"\n{occ} (total {score:.3f}):")
    ranked_dims = sorted(zip(DIMENSIONS, contribs), key=lambda x: -x[1])
    for dim, c in ranked_dims:
        print(f"  {dim:<40} {c:.3f}")


# --- Compare to a Sunyi-inclusive workplace scenario ---
# In a workplace where oral communication is done in BISINDO, the "oral
# communication" dimension effectively rescales - the candidate DOES have
# oral communication capacity, just in a different modality. Show what
# happens if we score oral communication as 0.75 to represent that scenario.
print()
print("=" * 78)
print("SUNYI-INCLUSIVE SCENARIO: oral communication rescaled")
print("=" * 78)
print("If the workplace communicates in BISINDO, oral comm effectively = 0.75")
print()
CANDIDATE_B_SUNYI = CANDIDATE_B_DEAF.copy()
CANDIDATE_B_SUNYI[5] = 0.75  # oral communication in signed workplace

scored2 = [(occ, cosine(CANDIDATE_B_SUNYI, v)) for occ, v in OCCUPATIONS.items()]
scored2.sort(key=lambda x: -x[1])
for i, (occ, score) in enumerate(scored2, 1):
    delta_score = score - dict(scored)[occ]
    print(f"  {i}. {occ:<28} {score:.3f}  ({delta_score:+.3f})")

print()
print("Rank movement (default -> Sunyi-inclusive):")
default_ranks = {o: i for i, (o, _) in enumerate(scored, 1)}
sunyi_ranks = {o: i for i, (o, _) in enumerate(scored2, 1)}
for occ in OCCUPATIONS:
    d = default_ranks[occ]
    s = sunyi_ranks[occ]
    if d != s:
        print(f"  {occ}: rank {d} -> {s}")
    else:
        print(f"  {occ}: rank {d} (unchanged)")


CANDIDATE B (Deaf) - strengths on the ten dimensions
  Analytical thinking                      0.45
  Creative thinking                        0.55
  Systems thinking                         0.40
  Resilience / flexibility                 0.85
  Empathy & active listening               0.85
  Communication (oral)                     0.30
  Communication (written)                  0.55
  Leadership & social influence            0.45
  Service orientation                      0.85
  Technological literacy                   0.45

RANKING
  1. Customer Service Rep         0.942
  2. IT Support Specialist        0.931
  3. Retail Sales Assistant       0.929
  4. Food Service Worker          0.922
  5. Software Developer           0.921
  6. Administrative Assistant     0.919
  7. Content Creator              0.912

PER-DIMENSION CONTRIBUTIONS FOR TOP 3

Customer Service Rep (total 0.942):
  Empathy & active listening               0.186
  Service orientation                      0.186
  Re

# Compute Vector

In [3]:
"""
Compute occupation vectors from O*NET Skills Importance ratings.

Uses the mapping documented in onet_mapping.md:
- For each dimension, take the MAX of the mapped O*NET skills' Importance values
- Rescale from 0-100 to 0-1
- Apply documented adjustments where noted

Prints both the vector and the intermediate O*NET values for verification.
"""

# O*NET Skills Importance data (0-100 scale) for each occupation
# Source: onetonline.org, fetched Aug 2026
ONET_DATA = {
    "Customer Service Rep": {  # 43-4051.00
        "Active Listening": 75, "Speaking": 72, "Reading Comprehension": 60,
        "Writing": 50, "Critical Thinking": 56, "Active Learning": 47,
        "Monitoring": 50, "Learning Strategies": 41, "Mathematics": 31, "Science": 0,
        "Service Orientation": 75, "Complex Problem Solving": 53, "Time Management": 53,
        "Negotiation": 50, "Persuasion": 50, "Social Perceptiveness": 50,
        "Coordination": 47, "Judgment and Decision Making": 47, "Instructing": 44,
        "Management of Personnel Resources": 35, "Systems Analysis": 31,
        "Systems Evaluation": 31, "Technology Design": 22,
        "Operations Analysis": 22, "Operations Monitoring": 0, "Troubleshooting": 0,
        "Programming": 0, "Quality Control Analysis": 0,
    },
    "Retail Sales Assistant": {  # 41-2031.00
        "Active Listening": 69, "Speaking": 69, "Reading Comprehension": 50,
        "Writing": 50, "Critical Thinking": 53, "Active Learning": 50,
        "Monitoring": 50, "Learning Strategies": 41, "Mathematics": 41, "Science": 0,
        "Persuasion": 72, "Service Orientation": 69, "Negotiation": 63,
        "Social Perceptiveness": 63, "Coordination": 50, "Time Management": 50,
        "Instructing": 47, "Judgment and Decision Making": 47,
        "Complex Problem Solving": 44, "Management of Personnel Resources": 35,
        "Systems Analysis": 25, "Systems Evaluation": 25, "Technology Design": 22,
        "Troubleshooting": 19, "Operations Analysis": 13, "Operations Monitoring": 25,
        "Programming": 6, "Quality Control Analysis": 25,
    },
    "Administrative Assistant": {  # 43-6014.00
        "Active Listening": 75, "Speaking": 75, "Reading Comprehension": 72,
        "Writing": 69, "Critical Thinking": 50, "Active Learning": 47,
        "Monitoring": 53, "Learning Strategies": 28, "Mathematics": 25, "Science": 0,
        "Service Orientation": 63, "Time Management": 63, "Coordination": 53,
        "Judgment and Decision Making": 50, "Social Perceptiveness": 50,
        "Complex Problem Solving": 47, "Management of Personnel Resources": 41,
        "Negotiation": 35, "Persuasion": 31, "Instructing": 28,
        "Systems Analysis": 25, "Systems Evaluation": 25,
        "Operations Analysis": 22, "Operations Monitoring": 22,
        "Quality Control Analysis": 22, "Programming": 16, "Technology Design": 13,
        "Troubleshooting": 0,
    },
    "Content Creator": {  # 27-3043.00 Writers and Authors
        "Writing": 94, "Reading Comprehension": 78, "Active Listening": 72,
        "Speaking": 72, "Critical Thinking": 66, "Active Learning": 60,
        "Monitoring": 50, "Learning Strategies": 44, "Mathematics": 6, "Science": 3,
        "Time Management": 66, "Coordination": 60, "Persuasion": 60,
        "Social Perceptiveness": 60, "Judgment and Decision Making": 56,
        "Service Orientation": 53, "Complex Problem Solving": 50,
        "Instructing": 47, "Negotiation": 47,
        "Management of Personnel Resources": 38, "Systems Evaluation": 35,
        "Systems Analysis": 31, "Technology Design": 19, "Programming": 16,
        "Quality Control Analysis": 13, "Operations Analysis": 6,
        "Operations Monitoring": 0, "Troubleshooting": 0,
    },
    "IT Support Specialist": {  # 15-1232.00
        "Active Listening": 75, "Reading Comprehension": 75, "Speaking": 75,
        "Critical Thinking": 69, "Writing": 63, "Active Learning": 53,
        "Learning Strategies": 53, "Monitoring": 53, "Mathematics": 31, "Science": 19,
        "Complex Problem Solving": 66, "Judgment and Decision Making": 56,
        "Service Orientation": 53, "Social Perceptiveness": 53, "Instructing": 50,
        "Operations Monitoring": 50, "Systems Analysis": 50, "Time Management": 50,
        "Troubleshooting": 50, "Coordination": 47, "Quality Control Analysis": 47,
        "Systems Evaluation": 47, "Persuasion": 41,
        "Management of Personnel Resources": 35, "Negotiation": 35,
        "Operations Analysis": 35, "Programming": 31, "Technology Design": 25,
    },
    "Food Service Worker": {  # 35-3031.00 Waiters and Waitresses
        "Service Orientation": 72, "Active Listening": 63, "Speaking": 63,
        "Social Perceptiveness": 60, "Coordination": 53, "Monitoring": 50,
        "Critical Thinking": 47, "Judgment and Decision Making": 44, "Persuasion": 44,
        "Reading Comprehension": 44, "Time Management": 44,
        "Complex Problem Solving": 41, "Active Learning": 38,
        "Quality Control Analysis": 31, "Instructing": 28, "Mathematics": 28,
        "Negotiation": 28, "Writing": 28, "Learning Strategies": 25,
        "Operations Monitoring": 25, "Systems Analysis": 25, "Systems Evaluation": 25,
        "Management of Personnel Resources": 22, "Troubleshooting": 19,
        "Programming": 3, "Operations Analysis": 0, "Science": 0, "Technology Design": 0,
    },
    "Software Developer": {  # 15-1252.00
        "Critical Thinking": 72, "Active Learning": 63, "Reading Comprehension": 63,
        "Active Listening": 60, "Writing": 56, "Speaking": 53, "Monitoring": 50,
        "Mathematics": 44, "Learning Strategies": 41, "Science": 28,
        "Programming": 75, "Judgment and Decision Making": 66,
        "Systems Analysis": 63, "Complex Problem Solving": 60, "Systems Evaluation": 60,
        "Technology Design": 60, "Operations Analysis": 56, "Time Management": 53,
        "Coordination": 50, "Instructing": 47, "Service Orientation": 47,
        "Social Perceptiveness": 47, "Operations Monitoring": 44, "Persuasion": 44,
        "Quality Control Analysis": 44, "Negotiation": 41, "Troubleshooting": 41,
        "Management of Personnel Resources": 31,
    },
}

# Mapping from ten dimensions to O*NET skills (max aggregation)
MAPPING = {
    "Analytical": ["Critical Thinking", "Complex Problem Solving",
                   "Judgment and Decision Making", "Mathematics"],
    "Creative": ["Operations Analysis", "Technology Design"],
    "Systems": ["Systems Analysis", "Systems Evaluation"],
    "Empathy": ["Active Listening", "Social Perceptiveness"],
    "OralComm": ["Speaking"],
    "WrittenComm": ["Writing", "Reading Comprehension"],
    "Leadership": ["Persuasion", "Negotiation",
                   "Management of Personnel Resources", "Instructing"],
    "Service": ["Service Orientation"],
    "TechLit": ["Programming", "Technology Design", "Troubleshooting",
                "Operations Monitoring"],
}

# Resilience values assigned from BLS narrative (see onet_mapping.md)
RESILIENCE = {
    "Customer Service Rep": 0.70,
    "Retail Sales Assistant": 0.65,
    "Administrative Assistant": 0.55,
    "Content Creator": 0.60,
    "IT Support Specialist": 0.65,
    "Food Service Worker": 0.75,
    "Software Developer": 0.55,
}

# Documented adjustments applied after O*NET mapping
ADJUSTMENTS = {
    "Content Creator": {
        "Creative": (0.75, "Indonesian entry-level content creator context "
                            "emphasizes creativity beyond what BLS Writers "
                            "and Authors captures"),
    },
}


def compute_vector(occupation_name):
    """Compute the ten-dimension vector for one occupation."""
    onet = ONET_DATA[occupation_name]
    vector = {}
    intermediate = {}

    for dim, skills in MAPPING.items():
        values = [onet.get(skill, 0) for skill in skills]
        max_value = max(values) if values else 0
        vector[dim] = round(max_value / 100.0, 2)
        intermediate[dim] = {"skills_used": {s: onet.get(s, 0) for s in skills},
                             "max": max_value, "rescaled": vector[dim]}

    # Insert resilience from separate table
    vector["Resilience"] = RESILIENCE[occupation_name]
    intermediate["Resilience"] = {"source": "BLS narrative, researcher judgment",
                                   "value": vector["Resilience"]}

    # Apply adjustments
    if occupation_name in ADJUSTMENTS:
        for dim, (new_val, reason) in ADJUSTMENTS[occupation_name].items():
            intermediate[dim]["adjusted_from"] = vector[dim]
            intermediate[dim]["adjusted_to"] = new_val
            intermediate[dim]["reason"] = reason
            vector[dim] = new_val

    return vector, intermediate


# Order: matches recommender.py dimension order
ORDER = ["Analytical", "Creative", "Systems", "Resilience", "Empathy",
         "OralComm", "WrittenComm", "Leadership", "Service", "TechLit"]

OCCUPATION_ORDER = [
    "Customer Service Rep",
    "Retail Sales Assistant",
    "Administrative Assistant",
    "Content Creator",
    "IT Support Specialist",
    "Food Service Worker",
    "Software Developer",
]


def main():
    print("=" * 80)
    print("O*NET-anchored occupation vectors")
    print("=" * 80)
    print()

    header = ["Occupation"] + ORDER
    widths = [24, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]
    row = "".join(h.ljust(w) for h, w in zip(header, widths))
    print(row)
    print("-" * sum(widths))

    print()
    print("Recommender-format vectors (Python literal):")
    print("OCCUPATIONS = {")

    for occ in OCCUPATION_ORDER:
        vec, _ = compute_vector(occ)
        values = [f"{vec[dim]:.2f}" for dim in ORDER]
        formatted = ", ".join(values)
        # Match the padding style of recommender.py
        name_field = f'    "{occ}":'.ljust(35)
        print(f'{name_field}[{formatted}],')

    print("}")
    print()

    # Also print the per-occupation intermediate for verification
    print("=" * 80)
    print("Intermediate values (for verification)")
    print("=" * 80)
    for occ in OCCUPATION_ORDER:
        vec, intermediate = compute_vector(occ)
        print()
        print(f"### {occ}")
        for dim in ORDER:
            info = intermediate[dim]
            if "skills_used" in info:
                skills_str = ", ".join(f"{s}={v}" for s, v in info["skills_used"].items())
                line = f"  {dim}: max({skills_str}) = {info['max']} -> {info['rescaled']}"
                if "adjusted_to" in info:
                    line += f" [ADJUSTED to {info['adjusted_to']}]"
                print(line)
            else:
                print(f"  {dim}: {info}")


if __name__ == "__main__":
    main()


O*NET-anchored occupation vectors

Occupation              AnalyticalCreativeSystems ResilienceEmpathy OralCommWrittenCommLeadershipService TechLit 
--------------------------------------------------------------------------------------------------------

Recommender-format vectors (Python literal):
OCCUPATIONS = {
    "Customer Service Rep":        [0.56, 0.22, 0.31, 0.70, 0.75, 0.72, 0.60, 0.50, 0.75, 0.22],
    "Retail Sales Assistant":      [0.53, 0.22, 0.25, 0.65, 0.69, 0.69, 0.50, 0.72, 0.69, 0.25],
    "Administrative Assistant":    [0.50, 0.22, 0.25, 0.55, 0.75, 0.75, 0.72, 0.41, 0.63, 0.22],
    "Content Creator":             [0.66, 0.75, 0.35, 0.60, 0.72, 0.72, 0.94, 0.60, 0.53, 0.19],
    "IT Support Specialist":       [0.69, 0.35, 0.50, 0.65, 0.75, 0.75, 0.75, 0.50, 0.53, 0.50],
    "Food Service Worker":         [0.47, 0.00, 0.25, 0.75, 0.63, 0.63, 0.44, 0.44, 0.72, 0.25],
    "Software Developer":          [0.72, 0.60, 0.63, 0.55, 0.60, 0.53, 0.63, 0.47, 0.47, 0.75],
}

In

# Test/Check

## Perturbation

In [4]:
"""Perturbation ablation.

For each of Candidate A and Candidate B, perturb every dimension by +0.05
and by -0.05 (clipped to [0, 1]) one at a time, re-rank the seven
occupations, and count how often the top-3 changes.

This tests whether the rankings reported in the Illustrative Example are
stable under small changes in the candidate's self-assessment values --
a natural robustness concern for a recommender whose input is a
self-report on a small number of dimensions.
"""
import numpy as np

def cosine_sim(a, b):
    """Cosine of the angle between two vectors. Returns a value in [0, 1] for
    non-negative vectors. Higher means the strengths profiles have their high
    values in similar places."""
    a_norm = l2_normalize(a)
    b_norm = l2_normalize(b)
    return sum(x * y for x, y in zip(a_norm, b_norm))

DIMS = [
    "Analytical", "Creative", "Systems", "Resilience",
    "Empathy & active listening", "Oral communication",
    "Written communication", "Leadership & social influence",
    "Service orientation", "Technological literacy",
]

CANDIDATE_A = np.array([0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.80, 0.35, 0.45, 0.80])
CANDIDATE_B = np.array([0.45, 0.55, 0.40, 0.85, 0.85, 0.30, 0.55, 0.45, 0.85, 0.45])


def rank(candidate):
    scores = {name: cosine_sim(candidate, np.array(vec))
              for name, vec in OCCUPATIONS.items()}
    return sorted(scores.items(), key=lambda kv: -kv[1])


def top_k(ranking, k=3):
    return tuple(name for name, _ in ranking[:k])


def perturbation_check(name, candidate, delta=0.05):
    base = rank(candidate)
    base_top1 = base[0][0]
    base_top3 = top_k(base, 3)
    print(f"\n=== {name} ===")
    print(f"Baseline top-1: {base_top1}")
    print(f"Baseline top-3: {base_top3}")
    print(f"\nPerturbing each dimension by \u00b1{delta} (clipped to [0,1]):\n")
    print(f"{'Dimension':<32}{'Delta':<8}{'Top-1':<28}{'Top-3 changed?':<15}")
    print("-" * 85)
    top1_changes = 0
    top3_changes = 0
    total = 0
    for i, dim in enumerate(DIMS):
        for sign in [+1, -1]:
            perturbed = candidate.copy()
            perturbed[i] = float(np.clip(candidate[i] + sign * delta, 0.0, 1.0))
            r = rank(perturbed)
            new_top1 = r[0][0]
            new_top3 = top_k(r, 3)
            top1_ch = new_top1 != base_top1
            top3_ch = new_top3 != base_top3
            if top1_ch:
                top1_changes += 1
            if top3_ch:
                top3_changes += 1
            total += 1
            flag = "yes" if top3_ch else "no"
            print(f"{dim:<32}{sign*delta:<+8.2f}{new_top1:<28}{flag:<15}")
    print(f"\nTotal perturbations tested: {total}")
    print(f"Perturbations that changed top-1: {top1_changes} / {total}")
    print(f"Perturbations that changed top-3 set: {top3_changes} / {total}")


if __name__ == "__main__":
    perturbation_check("Candidate A", CANDIDATE_A)
    perturbation_check("Candidate B (Deaf, general labour market)", CANDIDATE_B)


# Sunyi-inclusive scenario for Candidate B: oral communication rescaled to 0.75
CANDIDATE_B_SUNYI = CANDIDATE_B.copy()
CANDIDATE_B_SUNYI[5] = 0.75

if __name__ == "__main__":
    print("\n\n" + "=" * 85)
    print("Extra: Candidate B in the Sunyi-inclusive scenario")
    print("=" * 85)
    perturbation_check("Candidate B (Sunyi-inclusive scenario)", CANDIDATE_B_SUNYI)



=== Candidate A ===
Baseline top-1: Software Developer
Baseline top-3: ('Software Developer', 'IT Support Specialist', 'Content Creator')

Perturbing each dimension by ±0.05 (clipped to [0,1]):

Dimension                       Delta   Top-1                       Top-3 changed? 
-------------------------------------------------------------------------------------
Analytical                      +0.05   Software Developer          no             
Analytical                      -0.05   Software Developer          no             
Creative                        +0.05   Software Developer          no             
Creative                        -0.05   Software Developer          no             
Systems                         +0.05   Software Developer          no             
Systems                         -0.05   Software Developer          no             
Resilience                      +0.05   Software Developer          no             
Resilience                      -0.05   Softwa

## Robusness

In [5]:
"""
Robustness checks for the O*NET-anchored recommender.

Three simple checks:
1. Mapping sensitivity: max vs average aggregation for the dimensions
2. Null baseline: what does cosine similarity look like against random candidates
3. Content Creator adjustment: with vs without the 0.75 override

All checks report whether Candidate A and B rankings change.
"""

import random
random.seed(42)

CANDIDATE_A = [0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.80, 0.35, 0.45, 0.80]
# Candidate B (Deaf, hearing-default scoring) as used in the paper's
# Illustrative Example. Order matches ORDER in compute_vectors:
# [analytical, creative, systems, empathy, service, oral, written,
#  technological, resilience, leadership].
CANDIDATE_B = [0.45, 0.55, 0.40, 0.85, 0.85, 0.30, 0.55, 0.45, 0.85, 0.45]


def l2(v):
    m = math.sqrt(sum(x * x for x in v))
    return v if m == 0 else [x / m for x in v]


def cosine(a, b):
    an, bn = l2(a), l2(b)
    return sum(x * y for x, y in zip(an, bn))


def build_vectors(aggregator, apply_content_adjustment):
    """Build the seven occupation vectors given an aggregator (max or avg)
    and whether to apply the Content Creator Creative adjustment."""
    vectors = {}
    for occ in OCCUPATION_ORDER:
        onet = ONET_DATA[occ]
        vec = {}
        for dim, skills in MAPPING.items():
            vals = [onet.get(s, 0) for s in skills]
            if not vals:
                vec[dim] = 0.0
            elif aggregator == "max":
                vec[dim] = max(vals) / 100.0
            elif aggregator == "avg":
                vec[dim] = (sum(vals) / len(vals)) / 100.0
            else:
                raise ValueError(aggregator)
        vec["Resilience"] = RESILIENCE[occ]
        if apply_content_adjustment and occ == "Content Creator":
            vec["Creative"] = 0.75
        vectors[occ] = [round(vec[d], 3) for d in ORDER]
    return vectors


def ranking(candidate, vectors):
    scored = [(occ, cosine(candidate, v)) for occ, v in vectors.items()]
    scored.sort(key=lambda x: -x[1])
    return scored


def rank_string(scored):
    return " > ".join(f"{o.split()[0]}({s:.3f})" for o, s in scored)


def print_section(title):
    print()
    print("=" * 78)
    print(title)
    print("=" * 78)


# --- Baseline (current design: max, with adjustment) ---
baseline = build_vectors("max", apply_content_adjustment=True)
baseline_a = ranking(CANDIDATE_A, baseline)
baseline_b = ranking(CANDIDATE_B, baseline)

print_section("BASELINE: max aggregator, Content Creator Creative = 0.75")
print("Candidate A:", rank_string(baseline_a))
print("Candidate B:", rank_string(baseline_b))


# --- Check 1: Mapping sensitivity - average instead of max ---
print_section("CHECK 1: Average aggregator instead of max (with adjustment)")
avg_vectors = build_vectors("avg", apply_content_adjustment=True)
avg_a = ranking(CANDIDATE_A, avg_vectors)
avg_b = ranking(CANDIDATE_B, avg_vectors)
print("Candidate A:", rank_string(avg_a))
print("Candidate B:", rank_string(avg_b))

top3_stable_a = [o for o, _ in baseline_a[:3]] == [o for o, _ in avg_a[:3]]
top3_stable_b = [o for o, _ in baseline_b[:3]] == [o for o, _ in avg_b[:3]]
top1_stable_a = baseline_a[0][0] == avg_a[0][0]
top1_stable_b = baseline_b[0][0] == avg_b[0][0]

print()
print(f"Top-1 stable: A={top1_stable_a}, B={top1_stable_b}")
print(f"Top-3 (as set) stable: A={top3_stable_a}, B={top3_stable_b}")


# --- Check 2: Null baseline with random candidates ---
print_section("CHECK 2: Null baseline - random candidate similarity distribution")
N = 1000
random_sims = {occ: [] for occ in OCCUPATION_ORDER}
for _ in range(N):
    rand_cand = [random.random() for _ in range(10)]
    for occ, vec in baseline.items():
        random_sims[occ].append(cosine(rand_cand, vec))

print(f"Random candidates: N={N}, uniform [0,1] on each dimension")
print(f"{'Occupation':<28}{'Mean':<10}{'Std':<10}{'Min':<10}{'Max':<10}")
print("-" * 68)
for occ in OCCUPATION_ORDER:
    sims = random_sims[occ]
    mean = sum(sims) / len(sims)
    var = sum((s - mean) ** 2 for s in sims) / len(sims)
    std = math.sqrt(var)
    print(f"{occ:<28}{mean:<10.3f}{std:<10.3f}{min(sims):<10.3f}{max(sims):<10.3f}")

print()
print("Candidate A scores vs random-candidate mean (in std units):")
for occ, score in baseline_a:
    sims = random_sims[occ]
    mean = sum(sims) / len(sims)
    var = sum((s - mean) ** 2 for s in sims) / len(sims)
    std = math.sqrt(var) if var > 0 else 1
    z = (score - mean) / std
    print(f"  {occ:<28}score={score:.3f}  random_mean={mean:.3f}  z={z:+.2f}")

print()
print("Candidate B scores vs random-candidate mean (in std units):")
for occ, score in baseline_b:
    sims = random_sims[occ]
    mean = sum(sims) / len(sims)
    var = sum((s - mean) ** 2 for s in sims) / len(sims)
    std = math.sqrt(var) if var > 0 else 1
    z = (score - mean) / std
    print(f"  {occ:<28}score={score:.3f}  random_mean={mean:.3f}  z={z:+.2f}")


# --- Check 3: With vs without Content Creator adjustment ---
print_section("CHECK 3: Content Creator adjustment - with vs without")
noadj = build_vectors("max", apply_content_adjustment=False)
noadj_a = ranking(CANDIDATE_A, noadj)
noadj_b = ranking(CANDIDATE_B, noadj)
print("With adjustment (0.75):")
print("  Candidate A:", rank_string(baseline_a))
print("  Candidate B:", rank_string(baseline_b))
print()
print("Without adjustment (0.19 - raw O*NET):")
print("  Candidate A:", rank_string(noadj_a))
print("  Candidate B:", rank_string(noadj_b))

# Find Content Creator's rank in each
cc_baseline_a_rank = [o for o, _ in baseline_a].index("Content Creator") + 1
cc_noadj_a_rank = [o for o, _ in noadj_a].index("Content Creator") + 1
cc_baseline_b_rank = [o for o, _ in baseline_b].index("Content Creator") + 1
cc_noadj_b_rank = [o for o, _ in noadj_b].index("Content Creator") + 1

print()
print(f"Content Creator rank for Candidate A: with={cc_baseline_a_rank}, without={cc_noadj_a_rank}")
print(f"Content Creator rank for Candidate B: with={cc_baseline_b_rank}, without={cc_noadj_b_rank}")



BASELINE: max aggregator, Content Creator Creative = 0.75
Candidate A: Software(0.989) > IT(0.948) > Content(0.899) > Administrative(0.865) > Customer(0.861) > Retail(0.843) > Food(0.817)
Candidate B: Customer(0.942) > IT(0.931) > Retail(0.929) > Food(0.922) > Software(0.921) > Administrative(0.919) > Content(0.912)

CHECK 1: Average aggregator instead of max (with adjustment)
Candidate A: Software(0.981) > IT(0.932) > Content(0.865) > Administrative(0.841) > Retail(0.834) > Customer(0.822) > Food(0.773)
Candidate B: Software(0.933) > Retail(0.931) > IT(0.929) > Customer(0.925) > Content(0.908) > Food(0.907) > Administrative(0.901)

Top-1 stable: A=True, B=False
Top-3 (as set) stable: A=True, B=False

CHECK 2: Null baseline - random candidate similarity distribution
Random candidates: N=1000, uniform [0,1] on each dimension
Occupation                  Mean      Std       Min       Max       
--------------------------------------------------------------------
Customer Service Rep     

## Coverage Test, Distributional Skew Check, or Sensitivity Analysis

In [6]:
"""
Coverage Test, Distributional Skew Check, or Sensitivity Analysis for the strengths-based occupational recommender.

Runs the recommender against a diverse set of synthetic candidate profiles
and counts how often each occupation appears in the top-3 recommendations.
If the sample or method is biased, one or two occupations will dominate.
If it is reasonably balanced, each occupation should appear in the top-3
for candidates whose strengths match it.

This is not a validation of accuracy - only accuracy against real users
could validate that. This is a check on whether the design has an obvious
systematic tilt.
"""

import random
from math import comb


def rank(candidate, occupations):
    scores = [(name, cosine_similarity(candidate, vec)) for name, vec in occupations.items()]
    scores.sort(key=lambda x: -x[1])
    return [name for name, _ in scores]


# Twelve synthetic candidate profiles across a range of strengths patterns.
# Values are on 0-to-1 scale for the ten dimensions.
#
# NOTE: These twelve profiles are diagnostic tools for the sample coverage
# check reported in Methodology Section III.D. They are DISTINCT from the two
# candidate profiles used in the Illustrative Example (Section IV). In
# particular, "B: person-oriented" here is a legacy general-purpose profile,
# NOT the paper's Deaf Candidate B (which is defined in candidate_b_deaf.py
# with oral_comm=0.30 for hearing-default and 0.75 for Sunyi-inclusive).
CANDIDATES = {
    # Original two (diagnostic profiles, not the Illustrative Example candidates)
    "A: analytical/technical": [0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.80, 0.35, 0.45, 0.80],
    "B: person-oriented":      [0.40, 0.50, 0.40, 0.70, 0.90, 0.80, 0.55, 0.50, 0.85, 0.45],

    # Analytical variants
    "C: analytical, low tech":  [0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.80, 0.35, 0.45, 0.30],
    "D: analytical, low writing": [0.85, 0.55, 0.75, 0.50, 0.55, 0.45, 0.30, 0.35, 0.45, 0.80],

    # Person-oriented variants
    "E: empathy-strong, low oral": [0.40, 0.50, 0.40, 0.70, 0.90, 0.40, 0.55, 0.50, 0.85, 0.45],
    "F: service, low empathy":  [0.40, 0.50, 0.40, 0.70, 0.50, 0.80, 0.55, 0.50, 0.90, 0.45],

    # Creative
    "G: creative, moderate tech": [0.50, 0.90, 0.50, 0.55, 0.55, 0.60, 0.65, 0.40, 0.55, 0.70],

    # Balanced / no clear specialization
    "H: balanced, moderate all": [0.55, 0.55, 0.55, 0.60, 0.55, 0.60, 0.60, 0.50, 0.60, 0.55],

    # Communication-strong
    "I: communication, low tech": [0.55, 0.45, 0.50, 0.60, 0.70, 0.85, 0.85, 0.55, 0.60, 0.35],

    # Resilience-strong (life/practical experience)
    "J: resilient, practical":  [0.50, 0.55, 0.50, 0.90, 0.65, 0.65, 0.55, 0.55, 0.65, 0.50],

    # Leadership-strong (peer mentor, community role)
    "K: leadership-strong":     [0.60, 0.55, 0.60, 0.70, 0.70, 0.75, 0.65, 0.85, 0.65, 0.55],

    # Systems-strong (organizer, planner)
    "L: systems + written":     [0.65, 0.45, 0.85, 0.55, 0.45, 0.50, 0.80, 0.50, 0.50, 0.65],
}


def main():
    top1_counts = {name: 0 for name in OCCUPATIONS}
    top3_counts = {name: 0 for name in OCCUPATIONS}
    top1_appearances = {name: [] for name in OCCUPATIONS}

    print("=" * 78)
    print("BIAS CHECK: 12 candidate profiles against the 7-occupation sample")
    print("=" * 78)
    print()
    print(f"{'Candidate':<32}{'Top 1':<24}{'Top 2':<24}{'Top 3':<24}")
    print("-" * 104)

    for cand_name, vector in CANDIDATES.items():
        ranking = rank(vector, OCCUPATIONS)
        top1_counts[ranking[0]] += 1
        top1_appearances[ranking[0]].append(cand_name.split(":")[0])
        for occ in ranking[:3]:
            top3_counts[occ] += 1
        print(f"{cand_name:<32}{ranking[0]:<24}{ranking[1]:<24}{ranking[2]:<24}")

    print()
    print("=" * 78)
    print(f"How often each occupation appeared, out of {len(CANDIDATES)} candidates")
    print("=" * 78)
    print(f"{'Occupation':<28}{'Ranked #1':<14}{'In top 3':<14}{'Won for candidates':<30}")
    print("-" * 86)
    for occ in OCCUPATIONS:
        top1 = top1_counts[occ]
        top3 = top3_counts[occ]
        wins = ", ".join(top1_appearances[occ]) if top1_appearances[occ] else "-"
        print(f"{occ:<28}{top1:<14}{top3:<14}{wins:<30}")

    print()
    print("Interpretation guide:")
    print("- Balanced sample: each occupation appears in some candidate's top 3")
    print("- Biased sample: 1-2 occupations dominate top 1 across many candidates")
    print("- Never-top-3 occupations may need vector or sample review")

    # -------------------------------------------------------------------------
    # Input-responsiveness diagnostic vs random-ranker baseline
    #
    # A hostile reviewer's natural objection to a proof-of-concept recommender
    # is: "how do we know the rankings aren't just noise?"
    #
    # A minimum condition for the recommender to be doing meaningful work is
    # that different candidate strengths patterns produce different rankings.
    # If all 12 profiles produced the same top-3, the recommender would be
    # ignoring the input; if the 12 top-3s were random, that would be the
    # random-ranker outcome. Neither is what we observe.
    #
    # Two numbers below:
    # (i)  distinct top-3 sets produced by the recommender across the 12
    #      diagnostic profiles (higher = more input-responsive).
    # (ii) expected number of distinct top-3 sets a random ranker would
    #      produce across 12 trials, estimated by simulation. Under the null,
    #      each trial draws a uniform size-3 subset from C(7,3)=35.
    #
    # This is a necessary but not sufficient condition for meaningful work.
    # It does NOT validate accuracy against real user outcomes.
    # -------------------------------------------------------------------------
    N_TRIALS = 10000
    rng = random.Random(42)
    occ_list = list(OCCUPATIONS.keys())

    # (i) Recommender's distinct top-3 sets
    recommender_top3 = {
        cand_name: frozenset(rank(vector, OCCUPATIONS)[:3])
        for cand_name, vector in CANDIDATES.items()
    }
    distinct_top3s_recommender = len(set(recommender_top3.values()))

    # (ii) Random ranker: for each of N trials, draw 12 uniform size-3 subsets
    # and count distinct ones. Report the mean and empirical distribution.
    n_profiles = len(CANDIDATES)
    distinct_counts = []
    for _ in range(N_TRIALS):
        subs = [frozenset(rng.sample(occ_list, 3)) for _ in range(n_profiles)]
        distinct_counts.append(len(set(subs)))
    mean_distinct_random = sum(distinct_counts) / len(distinct_counts)
    # Empirical fraction of random trials with <= observed diversity
    p_le_observed = sum(1 for c in distinct_counts if c <= distinct_top3s_recommender) / len(distinct_counts)

    print()
    print("=" * 78)
    print("INPUT-RESPONSIVENESS DIAGNOSTIC (recommender vs random ranker)")
    print("=" * 78)
    print(f"Number of diagnostic profiles:                       {n_profiles}")
    print(f"Distinct top-3 sets from recommender:                {distinct_top3s_recommender} of {n_profiles}")
    print(f"Distinct top-3 sets from random ranker (mean, {N_TRIALS} trials): {mean_distinct_random:.2f}")
    print(f"Fraction of random trials with <= {distinct_top3s_recommender} distinct sets:      {p_le_observed:.4f}")
    print()
    print("Interpretation:")
    print(f"- The recommender collapses the 12 diagnostic profiles onto {distinct_top3s_recommender} distinct")
    print("  top-3 sets. This is fewer than a random ranker would produce")
    print(f"  (mean {mean_distinct_random:.2f}), reflecting that the recommender clusters similar")
    print("  strengths patterns onto similar occupational regions rather than")
    print("  producing random assignments.")
    print("- The recommender is not returning a single top-3 regardless of")
    print("  input; it is producing structured variation.")
    print("- This is a necessary but not sufficient condition for meaningful")
    print("  work. Accuracy against real user outcomes is not tested here")
    print("  and remains future work.")


if __name__ == "__main__":
    main()

BIAS CHECK: 12 candidate profiles against the 7-occupation sample

Candidate                       Top 1                   Top 2                   Top 3                   
--------------------------------------------------------------------------------------------------------
A: analytical/technical         Software Developer      IT Support Specialist   Content Creator         
B: person-oriented              Customer Service Rep    Administrative AssistantRetail Sales Assistant  
C: analytical, low tech         Software Developer      IT Support Specialist   Content Creator         
D: analytical, low writing      Software Developer      IT Support Specialist   Content Creator         
E: empathy-strong, low oral     Customer Service Rep    Retail Sales Assistant  IT Support Specialist   
F: service, low empathy         Customer Service Rep    Retail Sales Assistant  IT Support Specialist   
G: creative, moderate tech      Software Developer      Content Creator         IT Support Sp